In [ ]:
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/t1dbg'
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

In [ ]:
# verifica GPU
!nvidia-smi

In [ ]:
# Per installare rapids se non giÃ  disponibile
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

In [ ]:
import cuml
cuml.__version__

### Import librerie


In [ ]:
import os
import pandas as pd
import numpy as np
import json
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    mean_absolute_percentage_error,
)
import pickle
import lightgbm as lgb
import xgboost as xgb

from cuml.ensemble import RandomForestRegressor

from lib.data import load_splits, rescale_data, calculate_metrics, print_results, HYPO, HYPER, L_BOUND, U_BOUND, categorize_glucose

### Training Std_ML Utilities


In [ ]:
def create_model(model_type, seed):
    """Costruisci il modello specificato"""
    if model_type == "lgb":
        return lgb.LGBMRegressor(random_state=seed, device="gpu")
    elif model_type == "xgb":
        return xgb.XGBRegressor(random_state=seed, device="cuda:0")
    elif model_type == "rf":
       return RandomForestRegressor(random_state=seed)
    else:
        raise ValueError(f"Unsupported model type: {model_type}")

In [ ]:
def save_model(model, model_path):
    """Salva il modello addestrato"""
    with open(model_path, "wb") as handle:
        pickle.dump(model, handle, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Model saved to: {model_path}")

In [ ]:
def evaluate_and_save_results(model, val_set, X_cols, y_cols, output_path, exp_name):
    """Valuta e salva i risultati del modello"""
    # Inferisci
    val_set = val_set.copy()
    val_set["y_pred"] = model.predict(val_set[X_cols])

    # Prepara i risultati
    val_set = val_set.rename(columns={y_cols[-1]: "target"})
    val_set = rescale_data(val_set, ["target", "y_pred"])

    # Seleziona le colonne di output e salva
    output_columns = ["Timestamp", "Patient_ID", "bgClass", "target", "y_pred"]
    results = val_set[output_columns]

    print_results(results)

    output_file = f"{output_path}/{exp_name}_output.csv"
    results.to_csv(output_file, index=False)
    print(f"Results saved to: {output_file}")

    return results

### Main Pipeline


In [ ]:
# Configura il training
class Args:
    def __init__(self):
        self.output_path = "outputs/val_set"
        self.models_path = "models/val_set"
        self.exp_names = ["lgb", "xgb", "rf"]
        self.seed = 42


args = Args()

# Verifica i parametri
print(f"Configurazione:")
print(f"  - Models: {args.exp_names}")
print(f"  - Output path: {args.output_path}")
print(f"  - Models path: {args.models_path}")
print(f"  - Seed: {args.seed}")

In [ ]:
print(f"Starting ML training pipeline for models: {args.exp_names}...")

# Setup ambiente
os.makedirs(args.output_path, exist_ok=True)
os.makedirs(args.models_path, exist_ok=True)

In [ ]:
# Loading dei set dati
print("Loading pre-prepared data splits...")
train_set, val_set, test_set, X_cols, y_cols = load_splits()

In [ ]:
# Training loop per tutti i modelli
for exp_name in args.exp_names:
    print(f"\n{'='*60}")
    print(f"TRAINING {exp_name.upper()}")
    print(f"{'='*60}")

    # Costruisci e addestra il modello
    print(f"Creating and training {exp_name.upper()} model...")
    model = create_model(exp_name, args.seed)
    model.fit(train_set[X_cols], train_set[y_cols[-1]])

    # Salva il modello
    model_path = f"{args.models_path}/{exp_name}.pickle"
    save_model(model, model_path)

    # Valuta e salva i risultati
    results = evaluate_and_save_results(
        model, val_set, X_cols, y_cols, args.output_path, exp_name
    )

print(f"\nAll training pipelines completed successfully!")